## Step 1: カテゴリURLの抽出
トップページからカテゴリ候補URLを収集します。


In [4]:
import requests
from bs4 import BeautifulSoup

# サイトのトップページURL
BASE_URL = "https://keiei-manabu.com"

# トップページを取得してHTML解析
res = requests.get(BASE_URL)
soup = BeautifulSoup(res.content, "html.parser")

# カテゴリページのリンクを保存する集合（重複を防ぐためset）
category_links = set()

# ページ内のすべてのリンクを走査
for a in soup.find_all("a", href=True):
    href = a["href"]

    # カテゴリページとみなす条件を設定
    if href.startswith(BASE_URL + "/") \
        and len(href) < 50 \
        and "category" not in href \
        and BASE_URL + "/page/" not in href \
        and href != BASE_URL + "/":

        category_links.add(href)

# ソートして見やすく表示
category_links = sorted(list(category_links))

# 結果出力
print(f"カテゴリの総数: {len(category_links)}")
print("カテゴリ一覧:")
for url in category_links:
    print(url)


カテゴリの総数: 35
カテゴリ一覧:
https://keiei-manabu.com/accounting
https://keiei-manabu.com/advertising/
https://keiei-manabu.com/business-plan
https://keiei-manabu.com/business/
https://keiei-manabu.com/contentbusiness/
https://keiei-manabu.com/copywriting/
https://keiei-manabu.com/criticalthinking
https://keiei-manabu.com/economics
https://keiei-manabu.com/entrepreneur/
https://keiei-manabu.com/finance
https://keiei-manabu.com/finance/factoring
https://keiei-manabu.com/game-theory
https://keiei-manabu.com/humanresources
https://keiei-manabu.com/internetbusiness/
https://keiei-manabu.com/keieinokiso
https://keiei-manabu.com/lanchesterstrategy/
https://keiei-manabu.com/leadership
https://keiei-manabu.com/legal
https://keiei-manabu.com/mailmagazine/
https://keiei-manabu.com/mailmagazinetouroku.html
https://keiei-manabu.com/marketing
https://keiei-manabu.com/mindset/
https://keiei-manabu.com/otoiawase.html
https://keiei-manabu.com/presentation
https://keiei-manabu.com/profile.html
https://keiei-man

## Step 2: 不要なカテゴリの除外（手動フィルタリング）
プロフィールやお問い合わせなど不要なカテゴリを除外。


In [7]:
# 除外対象のURLリスト（記事とは無関係なページを手動で選定）
# 例：プロフィール、メルマガ登録、セミナー案内など
exclude_urls = [
    "https://keiei-manabu.com/mailmagazinetouroku.html",
    "https://keiei-manabu.com/profile.html",
    "https://keiei-manabu.com/videoseminar",
    "https://keiei-manabu.com/toolkyouzai",
    "https://keiei-manabu.com/sitemap.html",
    "https://keiei-manabu.com/realbusiness",
    "https://keiei-manabu.com/psychology",
    "https://keiei-manabu.com/otoiawase.html",
    "https://keiei-manabu.com/mailmagazine",
    "https://keiei-manabu.com/internetbusiness",
]

# URL末尾のスラッシュ有無を統一するための関数（正規化）
def normalize_url(url):
    return url.rstrip('/')

# 正規化した除外URLをセットに変換（検索高速化のため）
exclude_urls_normalized = set(normalize_url(url) for url in exclude_urls)

# 元のカテゴリリストから、除外対象を取り除いたフィルタ済みリストを作成
category_links_filtered = [
    url for url in category_links
    if normalize_url(url) not in exclude_urls_normalized
]

# フィルタ後のカテゴリ一覧を出力
print(f"除外後カテゴリの総数: {len(category_links_filtered)}")
print("除外後のカテゴリ一覧:")
for url in category_links_filtered:
    print(url)

除外後カテゴリの総数: 25
除外後のカテゴリ一覧:
https://keiei-manabu.com/accounting
https://keiei-manabu.com/advertising/
https://keiei-manabu.com/business-plan
https://keiei-manabu.com/business/
https://keiei-manabu.com/contentbusiness/
https://keiei-manabu.com/copywriting/
https://keiei-manabu.com/criticalthinking
https://keiei-manabu.com/economics
https://keiei-manabu.com/entrepreneur/
https://keiei-manabu.com/finance
https://keiei-manabu.com/finance/factoring
https://keiei-manabu.com/game-theory
https://keiei-manabu.com/humanresources
https://keiei-manabu.com/keieinokiso
https://keiei-manabu.com/lanchesterstrategy/
https://keiei-manabu.com/leadership
https://keiei-manabu.com/legal
https://keiei-manabu.com/marketing
https://keiei-manabu.com/mindset/
https://keiei-manabu.com/presentation
https://keiei-manabu.com/proposal
https://keiei-manabu.com/sales/
https://keiei-manabu.com/statistics
https://keiei-manabu.com/strategy
https://keiei-manabu.com/videomarketing/


## Step 3: 記事の本文を自動取得
カテゴリごとに記事リンクをたどり、本文を抽出・CSV保存します。


In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv

# 対象とするカテゴリページのURL一覧（手動で選定）
# これらのカテゴリから記事リンクを収集します
category_links_filtered = [
    "https://keiei-manabu.com/accounting",
    "https://keiei-manabu.com/advertising/",
    "https://keiei-manabu.com/business-plan",
    ...
]

#  全記事URLを保存するセット（重複排除のためsetを使用）
all_article_urls = set()

# 各カテゴリページから記事リンクを収集
for cat_url in category_links_filtered:
    res = requests.get(cat_url)
    soup = BeautifulSoup(res.content, "html.parser")

    # ページ内のすべてのリンクを走査
    for a in soup.find_all("a", href=True):
        href = a["href"]

        # 「.html」で終わるものを記事と見なす（静的記事ページ）
        if href.endswith(".html"):
            if href.startswith("http"):
                url = href
            elif href.startswith("/"):
                url = "https://keiei-manabu.com" + href
            else:
                url = cat_url.rstrip("/") + "/" + href

            # 同一カテゴリ内に限定（カテゴリ外へのリンクを除外）
            if url.startswith(cat_url):
                all_article_urls.add(url)

    time.sleep(0.2)  # サーバー負荷軽減のため待機

print(f"全カテゴリ合計の記事数: {len(all_article_urls)}")

# 各記事からタイトルと本文を抽出して保存
articles = []

for i, url in enumerate(all_article_urls):
    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.content, "html.parser")

        # タイトル抽出
        title_tag = soup.find("title")
        title = title_tag.get_text(strip=True) if title_tag else ""

        #  本文の抽出（本文はid="text1"のdivタグ内にある）
        main_div = soup.find("div", id="text1")
        if main_div:
            paragraphs = main_div.find_all("p")
            text = "\n".join([p.get_text(strip=True) for p in paragraphs])

            # 本文がある程度の長さがある場合のみ採用
            if text and len(text) > 50:
                articles.append({
                    "url": url,
                    "title": title,
                    "body": text
                })
            else:
                print(f"本文が短い or 空記事: {url}")
        else:
            print(f"本文（id=text1）が見つからない: {url}")
    except Exception as e:
        print(f"Error at {url}: {e}")

    time.sleep(0.5)  # サーバーに優しい間隔

    # 進捗表示（10記事ごとに通知）
    if (i+1) % 10 == 0:
        print(f"{i+1}記事処理完了")

# 最終的な取得記事数とサンプル出力
print(f"\n最終的に取得できた記事数: {len(articles)}")
for article in articles[:2]:  # サンプルとして最初の2件のみ表示
    print(f"\nURL: {article['url']}")
    print(f"タイトル: {article['title']}")
    print(f"本文（冒頭300字）:\n{article['body'][:300]}")

# 記事データをCSVとして保存
with open("keiei_articles.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["url", "title", "body"])
    writer.writeheader()
    for article in articles:
        writer.writerow(article)

# Google Colab上でCSVをローカルにダウンロードできるようにする
from google.colab import files
files.download("keiei_articles.csv")
